# Gurjanov 28-day generation-core query

## Goal

Run the bundled 48-endpoint generation-core query and turn its sparse daily result into two
useful tables: longitudinal body weight and terminal laboratory values.

The query keeps factual source days. The terminal day range below is a display
choice for this 28-day study example; it does not modify the query result.

In [ ]:
import os
from pathlib import Path

source_from_environment = os.environ.get("VCGEN_SEND_ARCHIVE")
if source_from_environment:
    SEND_SOURCE = Path(source_from_environment)
else:
    SEND_SOURCE = next(
        (
            root / "rodent_db_export.zip"
            for root in (Path.cwd(), *Path.cwd().parents)
            if (root / "rodent_db_export.zip").is_file()
        ),
        Path("rodent_db_export.zip"),
    )

EXTRACTION_CACHE = Path(
    os.environ.get("VCGEN_QUERY_CACHE", SEND_SOURCE.parent / ".vcgen_query_cache")
)

if not SEND_SOURCE.exists():
    raise FileNotFoundError(
        "Set VCGEN_SEND_ARCHIVE to a SEND ZIP archive or extracted CSV directory; "
        f"got {SEND_SOURCE}"
    )

## 1. Run the bundled query

The package includes the strict generation-core query used by the repository
pipeline. This query is distinct from the broad Wistar Han discovery query.

In [ ]:
from importlib.resources import as_file, files

import vcgen_query as vq

query_resource = files("vcgen_query").joinpath(
    "data/queries/gurjanov_28d_generation_core.yaml"
)
with as_file(query_resource) as query_path:
    query = vq.load_query(query_path)

result: vq.QueryResult = vq.execute_query(
    SEND_SOURCE,
    query,
    extract_dir=EXTRACTION_CACHE,
)

{
    "query_id": query.query_id,
    "subjects": len(result.subjects),
    "studies": result.subjects["study_id"].nunique(),
    "requested_endpoints": len(query.requested_endpoints),
}

In [ ]:
result.subjects[
    ["study_id", "subject_id", "sex", "study_year", "study_length_days"]
].head()

## 2. Index the daily values

`DailyFrame.values` has one row for each observed subject-day. Most endpoint
cells are empty because collection schedules differ.

In [ ]:
daily = vq.to_daily_frame(result, on_conflict="report")
KEY_COLUMNS = ["study_id", "subject_id", "study_day"]

daily_values = daily.values.set_index(KEY_COLUMNS).sort_index()
daily_values[["BW"]].dropna().head(8)

## 3. Select terminal laboratory rows

Keep laboratory columns, restrict the example to days 21–35, and remove rows
where none of those columns was measured.

In [ ]:
TERMINAL_DAY_RANGE = (21, 35)
lab_columns = [
    facts.endpoint_name
    for facts in result.endpoint_facts
    if facts.domain == "LB"
]

terminal_lab_rows = (
    daily.values.loc[
        daily.values["study_day"].between(*TERMINAL_DAY_RANGE),
        [*KEY_COLUMNS, *lab_columns],
    ]
    .dropna(subset=lab_columns, how="all")
    .set_index(KEY_COLUMNS)
    .sort_index()
)

terminal_lab_rows.iloc[:5, :8]

## 4. Reshape to tidy long form

Long form is convenient for grouping, plotting, and joining endpoint metadata.

In [ ]:
terminal_lab_long = (
    terminal_lab_rows.reset_index()
    .melt(
        id_vars=KEY_COLUMNS,
        value_vars=lab_columns,
        var_name="endpoint_name",
        value_name="value",
    )
    .dropna(subset=["value"])
    .sort_values([*KEY_COLUMNS, "endpoint_name"])
    .reset_index(drop=True)
)

terminal_lab_long.head(12)

In [ ]:
{
    "daily_rows": len(daily_values),
    "terminal_lab_rows": len(terminal_lab_rows),
    "terminal_lab_values": len(terminal_lab_long),
    "terminal_day_min": terminal_lab_long["study_day"].min(),
    "terminal_day_max": terminal_lab_long["study_day"].max(),
    "reported_conflicts": daily.receipt["conflict_count"],
}

## Takeaway

- Use the indexed wide table for subject trajectories.
- Select endpoint columns and then drop empty rows to inspect sparse panels.
- Use the long table for summaries and plots.
- Change `TERMINAL_DAY_RANGE` when a study uses a different terminal schedule.

The full factual records, states, source units, and audit fields remain available
in `result.observations`, `daily.states`, and `daily.cell_audit`.